<div dir="rtl">

# الدرس 32: التنظيم: معاقبة الأوزان الكبيرة

دفتر الكود لهذا الدرس. شغّل الخلايا بالترتيب من الأعلى.

</div>

<div dir="rtl">

### كثيرة حدود من الدرجة 15، مع تنظيم وبدونه

</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge

rng = np.random.default_rng(1)
f = lambda x: np.sin(2.6 * x) * 1.2 + 0.4 * x
x = np.sort(rng.uniform(-1, 1, 15)); y = f(x) + rng.normal(0, 0.28, 15)
xv = rng.uniform(-1, 1, 200); yv = f(xv) + rng.normal(0, 0.28, 200)

xx = np.linspace(-1, 1, 400)
plt.figure(figsize=(6.4, 3.8))
plt.scatter(x, y, color="C1", zorder=3, label="train points")
for alpha in [None, 1e-4, 1e-2, 10]:
    reg = LinearRegression() if alpha is None else Ridge(alpha=alpha)
    m = make_pipeline(PolynomialFeatures(15, include_bias=False), StandardScaler(), reg).fit(x[:, None], y)
    val_mse = np.mean((m.predict(xv[:, None]) - yv) ** 2)
    biggest = np.abs(m[-1].coef_).max()
    print(f"alpha = {str(alpha):6}  largest weight = {biggest:12.1f}   validation MSE = {val_mse:12.3f}")
    plt.plot(xx, m.predict(xx[:, None]), lw=2, label=f"alpha = {alpha}")
plt.ylim(-3, 3); plt.legend(fontsize=8); plt.grid(True)
plt.show()

<div dir="rtl">

### عقوبة القيم المطلقة تختار الميزات

</div>

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso
from sklearn.metrics import r2_score

data = load_diabetes()
X_tr, X_te, y_tr, y_te = train_test_split(data.data, data.target, test_size=0.25, random_state=0)
scaler = StandardScaler().fit(X_tr)
Xs_tr, Xs_te = scaler.transform(X_tr), scaler.transform(X_te)

for alpha in [0.01, 1, 3, 10]:
    lasso = Lasso(alpha=alpha).fit(Xs_tr, y_tr)
    kept = [n for n, c in zip(data.feature_names, lasso.coef_) if c != 0]
    print(f"alpha = {alpha:<5} features kept = {len(kept):2d}   test R2 = {r2_score(y_te, lasso.predict(Xs_te)):.3f}   {kept}")

<div dir="rtl">

### مسار الأوزان

</div>

In [ ]:
alphas = np.logspace(-2, 2, 60)
coefs = np.array([Lasso(alpha=a).fit(Xs_tr, y_tr).coef_ for a in alphas])

plt.figure(figsize=(6.4, 3.6))
for j in range(coefs.shape[1]):
    plt.plot(alphas, coefs[:, j], lw=1.8, label=data.feature_names[j])
plt.xscale("log"); plt.axhline(0, color="gray", lw=1)
plt.xlabel("alpha (log scale)"); plt.ylabel("weight")
plt.legend(fontsize=7, ncol=2)
plt.show()

<div dir="rtl">

### نختار القوة بالتحقق المتقاطع

</div>

In [ ]:
from sklearn.linear_model import RidgeCV, LassoCV

ridge = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 50))).fit(X_tr, y_tr)
lasso = make_pipeline(StandardScaler(), LassoCV(cv=5, random_state=0)).fit(X_tr, y_tr)
print("ridge best alpha:", round(ridge[-1].alpha_, 3), " test R2:", round(ridge.score(X_te, y_te), 3))
print("lasso best alpha:", round(lasso[-1].alpha_, 3), " test R2:", round(lasso.score(X_te, y_te), 3))

<div dir="rtl">

## تمرين: وزن يضمحل

وزن قيمته 2. البيانات لا تدفعه أبداً: مشتقة الخطأ صفر دائماً. معدل التعلّم 0.1، وقوة التنظيم 0.5.

1. مع عقوبة المربعات: بكم يُضرب الوزن في كل خطوة؟ وما قيمته بعد 3 خطوات؟
2. مع عقوبة القيم المطلقة: كم ينقص في كل خطوة؟ وبعد كم خطوة يصل إلى الصفر؟
3. اكتب حلقة بايثون تحاكي الحالتين 50 خطوة، واطبع القيمة النهائية. للثانية: إذا عبر الوزن الصفر، اجعله صفراً.

</div>

In [ ]:
# اكتب حلّك هنا
